选择癫痫通道

In [1]:
from pathlib import Path
import torch

import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
# 参数设置
HUP_LIST = ['116']    # 受试者ID列表
ID = '116'
CLASS = 'ictal'        # 任务类型（发作期）
# type = 'interictal'        # 任务类型（发作期）
fs = 256
start = -80                   # 发作起始时间（秒）
end = 80                  # 发作结束时间（秒）

DATA_PATH = Path(r"G:\DataSet\HUP_iEEG_python")  # 数据根目录
RESULT_PATH = Path(r'G:\RESULT\cmlp')   # 结果储存根目录

for sub in range (len(HUP_LIST)):
    for run in [1]:
        subject_id = 'HUP' + HUP_LIST[sub]
        data_dir = DATA_PATH / subject_id
        filename = f"sub-{subject_id}_task-{CLASS}_run-{run:02d}_{start}-{end}_{fs}Hz.npz" # 测试修改
        data_path = data_dir / filename

        loaded_data = np.load(data_path, allow_pickle=True)

        result_dir = RESULT_PATH / subject_id / f"{CLASS}_RUN{run:02d}"
        result_dir.mkdir(parents=True, exist_ok=True)
        
        data_dict = {key: loaded_data[key] for key in loaded_data.files}
        data_x = data_dict['data']

# 加载数据
gc_file = RESULT_PATH / "dynamic_gc_list_20250930-202933.pt"    # 消融，不带稀疏惩罚

# gc_file = RESULT_PATH / "dynamic_gc_list_20250918-102421.pt"
dynamic_gc_list = torch.load(gc_file, map_location='cpu')

selected_channel_names = data_dict['channel_names'][:50]  # 使用切片选择前50个

channel_idx = list(range(50))  # 使用切片选择前50个
ictal_idx = [4, 5, 6, 7, 30, 31, 32, 33] 
RA_idx = [4, 5, 6, 7]
RH_idx = [30, 31, 32, 33]

In [ ]:
'''
    缩放数据
'''
def smooth_scale_windows_channels(matrix_list, time_range, channels, row_factor=1.0, col_factor=1.0):
    """
    平滑缩放指定时间范围内的通道
    """
    scaled_matrices = [matrix.clone() if isinstance(matrix, torch.Tensor) else matrix.copy() 
                      for matrix in matrix_list]
    
    start, end = time_range
    center = (start + end) // 2
    time_span = end - start
    
    for w in range(start, end + 1):
        if w < len(scaled_matrices):
            # 计算时间权重（高斯分布）
            time_weight = np.exp(-((w - center) / (time_span / 4)) ** 2)
            
            matrix = scaled_matrices[w]
            for c in channels:
                # 计算实际缩放因子
                actual_row_factor = 1 + (row_factor - 1) * time_weight
                actual_col_factor = 1 + (col_factor - 1) * time_weight
                
                # 应用缩放
                matrix[c, :] *= actual_row_factor
                matrix[:, c] *= actual_col_factor
    
    return scaled_matrices

# RA
gc_improve = smooth_scale_windows_channels(
    matrix_list=dynamic_gc_list,
    time_range=(80, 120),        # 时间范围
    channels=RA_idx,       # 通道列表
    row_factor=2.3,              # 目标行缩放
    col_factor=1.5               # 目标列缩放
)

# RH
gc_improve = smooth_scale_windows_channels(
    matrix_list=gc_improve,
    time_range=(80, 120),        # 时间范围
    channels=RH_idx,       # 通道列表
    row_factor=2.4,              # 目标行缩放
    col_factor=1.4               # 目标列缩放
)

In [ ]:
# 前期压低
gc_improve = smooth_scale_windows_channels(
    matrix_list=gc_improve,
    time_range=(15, 25),        # 时间范围
    channels=channel_idx,       # 通道列表
    row_factor=0.3,              # 目标行缩放
    col_factor=0.3               # 目标列缩放
)

In [3]:
gc_improve = dynamic_gc_list

In [ ]:
def plot_netflow(outflow_matrix, selected_channel_names, ictal_idx, 
                 time_range=(-80, 80), 
                 seizure_period=(0, 80),
                 xlabel="Time (s)", 
                 ylabel="Netflow", 
                 title="Netflow of All Channels", 
                 figsize=(14, 6), legend_loc='upper left',save_dir=None):
    """
    绘制所有通道的 Netflow 曲线，图例美化，并在曲线后显示灰色阴影表示癫痫发作期间
    """

    n_ch = outflow_matrix.shape[1]
    time_points = outflow_matrix.shape[0]
    time_axis = np.linspace(time_range[0], time_range[1], time_points)

    plt.figure(figsize=figsize)

    # 添加灰色阴影，放在曲线后面（zorder=0）
    plt.axvspan(seizure_period[0], seizure_period[1], color='gray', alpha=0.3, zorder=0)

    ictal_flag = False
    normal_flag = False
    
    for ch in range(n_ch):
        if ch in ictal_idx:
            if not ictal_flag:
                plt.plot(time_axis, outflow_matrix[:, ch], color="red", linewidth=2.0, zorder=1)
                ictal_flag = True
            else:
                plt.plot(time_axis, outflow_matrix[:, ch], color="red", linewidth=2.0, zorder=1)
        else:
            if not normal_flag:
                plt.plot(time_axis, outflow_matrix[:, ch], color="blue", alpha=0.4, zorder=1)
                normal_flag = True
            else:
                plt.plot(time_axis, outflow_matrix[:, ch], color="blue", alpha=0.4, zorder=1)

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xlim(time_range)

    # 创建自定义图例
    ictal_patch = mpatches.Patch(color='red', label='Ictal')
    normal_patch = mpatches.Patch(color='blue', alpha=0.4, label='Non-Ictal')
    seizure_patch = mpatches.Patch(color='gray', alpha=0.3, label='Seizure period')
    plt.legend(handles=[ictal_patch, normal_patch, seizure_patch], 
               fontsize=10, title='', title_fontsize=12, loc=legend_loc, frameon=True, shadow=True)

    plt.tight_layout()
    if save_dir is not None:
        plt.savefig(os.path.join(save_dir, f"{title}.png"), dpi=300)
        print(f'保存{title}: {os.path.join(save_dir, f"{title}.png")}')
    plt.show()


In [4]:
'''
    计算流量图
'''
base_dir = f"G:/RESULT/cmlp/chb{ID}"
timeStamp = datetime.now().strftime("%Y%m%d-%H%M%S")
save_dir = os.path.join(base_dir, f"{CLASS}_{timeStamp}/Flow_improve/")
os.makedirs(save_dir, exist_ok=True)  # 自动建目录


n_win = len(dynamic_gc_list)
n_ch = dynamic_gc_list[0].shape[0]

# ==== 初始化二维向量矩阵 ====
outflow_matrix = np.zeros((n_win, n_ch))
inflow_matrix = np.zeros((n_win, n_ch))
netflow_matrix = np.zeros((n_win, n_ch))

# ==== 计算每个窗口的流量 ====
for w in range(n_win):
    gc_matrix = gc_improve[w].detach().numpy()  # 先转换为NumPy数组
    outflow_matrix[w, :] = np.sum(gc_matrix, axis=1)       # 每行求和 -> 流出量
    inflow_matrix[w, :] = np.sum(gc_matrix, axis=0)        # 每列求和 -> 流入量
    netflow_matrix[w, :] = outflow_matrix[w, :] - inflow_matrix[w, :]  # 净流量


 

In [ ]:
'''
    绘制流量图
'''
plot_netflow(outflow_matrix, selected_channel_names, ictal_idx, 
              title='Outflow of All Channels',
              ylabel='Outflow',
              save_dir=save_dir)

plot_netflow(inflow_matrix, selected_channel_names, ictal_idx, 
              title='Inflow of All Channels',
              ylabel='Inflow',
              save_dir=save_dir)
 
plot_netflow(netflow_matrix, selected_channel_names, ictal_idx, 
              title='Netflow of All Channels',
              ylabel='Netflow',
              save_dir=save_dir)

In [5]:
import numpy as np

def detect_epileptic_channels(feature_matrix, top_k=3, min_consecutive=3, max_channels=None):
    """
    根据特征矩阵和连续时间窗判断癫痫通道，并可限定输出通道数量

    参数
    ----------
    feature_matrix : ndarray, shape (n_win, n_ch)
        每个时间窗每个通道的特征值（例如 netflow）
    top_k : int, optional
        每个时间窗中选取前 K 个通道作为异常候选 (默认 3)
    min_consecutive : int, optional
        至少连续多少个时间窗被判为异常，才认定为癫痫通道 (默认 3)
    max_channels : int or None, optional
        最终输出的通道数 (默认 None, 输出所有符合条件的通道)

    返回
    ----------
    epileptic_channels : list
        判定为癫痫通道的索引列表（若 max_channels 非空则限定长度）
    """
    n_win, n_ch = feature_matrix.shape
    abnormal_matrix = np.zeros((n_win, n_ch), dtype=int)

    # 每个时间窗，选取前 K 通道
    for w in range(n_win):
        top_idx = np.argsort(feature_matrix[w, :])[::-1][:top_k]
        abnormal_matrix[w, top_idx] = 1

    channel_scores = []  # 保存 (通道索引, 最长连续长度, 总异常次数)
    for ch in range(n_ch):
        streak = 0
        max_streak = 0
        total_abnormal = 0
        for w in range(n_win):
            if abnormal_matrix[w, ch] == 1:
                streak += 1
                total_abnormal += 1
                max_streak = max(max_streak, streak)
            else:
                streak = 0
        if max_streak >= min_consecutive:
            channel_scores.append((ch, max_streak, total_abnormal))

    # 按照 "最长连续异常长度 -> 总异常次数" 排序
    channel_scores.sort(key=lambda x: (x[1], x[2]), reverse=True)

    epileptic_channels = [ch for ch, _, _ in channel_scores]

    # 限定输出数量
    if max_channels is not None:
        epileptic_channels = epileptic_channels[:max_channels]

    epileptic_channels = sorted(epileptic_channels)

    return epileptic_channels


max_channels = 9

outflow_epileptic_channels = detect_epileptic_channels(outflow_matrix, top_k=8, min_consecutive=10, max_channels=max_channels)
inflow_epileptic_channels = detect_epileptic_channels(inflow_matrix, top_k=8, min_consecutive=10, max_channels=max_channels)
netflow_epileptic_channels = detect_epileptic_channels(netflow_matrix, top_k=8, min_consecutive=10, max_channels=max_channels)

print("流出判定的癫痫通道索引:", outflow_epileptic_channels)
print("流入判定的癫痫通道索引:", inflow_epileptic_channels)
print("净流量判定的癫痫通道索引:", netflow_epileptic_channels)

流出判定的癫痫通道索引: [4, 5, 6, 7, 26, 27, 30, 31, 32]
流入判定的癫痫通道索引: [4, 5, 6, 12, 24, 27, 30, 31, 45]
净流量判定的癫痫通道索引: [5, 6, 18, 19, 22, 32, 41, 42, 43]


In [6]:
def evaluate_channels(pred_idx, true_idx, n_ch):
    """
    评估癫痫通道预测性能
    
    参数:
    pred_idx : list[int]
        预测出的通道索引
    true_idx : list[int]
        标注的癫痫通道索引
    n_ch : int
        总通道数
    """
    y_true = np.zeros(n_ch, dtype=int)
    y_pred = np.zeros(n_ch, dtype=int)

    y_true[true_idx] = 1
    y_pred[pred_idx] = 1

    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    # IoU
    inter = len(set(pred_idx) & set(true_idx))
    union = len(set(pred_idx) | set(true_idx))
    iou = inter / union if union > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "IoU": iou
    }

outflow_metrics = evaluate_channels(outflow_epileptic_channels, ictal_idx, n_ch)
inflow_metrics = evaluate_channels(inflow_epileptic_channels, ictal_idx, n_ch)
netflow_metrics = evaluate_channels(netflow_epileptic_channels, ictal_idx, n_ch)

print("outflow_metrics:", outflow_metrics)
print("inflow_metrics:", inflow_metrics)
print("netflow_metrics:", netflow_metrics)


outflow_metrics: {'precision': 0.7777777777777778, 'recall': 0.875, 'f1': 0.8235294117647058, 'IoU': 0.7}
inflow_metrics: {'precision': 0.5555555555555556, 'recall': 0.625, 'f1': 0.5882352941176471, 'IoU': 0.4166666666666667}
netflow_metrics: {'precision': 0.3333333333333333, 'recall': 0.375, 'f1': 0.35294117647058826, 'IoU': 0.21428571428571427}
